In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from deepseti.injection.dataset import load_split_frame_pool
from deepseti.preprocessing.frames import normalize_frames
from deepseti.models.vit_mae import (
    patchify,
    unpatchify,
    PatchEmbedding,
    PositionalEmbedding,
    random_masking,
)

PROJECT_ROOT = Path("..")

device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", device)

In [ ]:
raw_train, train_metadata = load_split_frame_pool(
    manifest_path="../configs/data/observations.csv",
    raw_root="../data/raw",
    split="train",
    f_start=2200,
    f_stop=2300,
)

train_frames = normalize_frames(raw_train)

print("Training frames:", train_frames.shape)

print(
    "Observations:",
    len({
        item["observation_id"]
        for item in train_metadata
    }),
)

In [ ]:
%matplotlib inline
torch.manual_seed(42)

# Take one real normal training frame
frame = torch.tensor(
    train_frames[0],
    dtype=torch.float32,
).unsqueeze(0).unsqueeze(0)

print("Frame shape:", frame.shape)


# Convert patches into transformer tokens
patch_embed = PatchEmbedding(
    patch_size=(4, 64),
    embed_dim=128,
)

position_embed = PositionalEmbedding(
    frame_shape=(16, 4096),
    patch_size=(4, 64),
    embed_dim=128,
)

tokens = patch_embed(frame)
tokens = position_embed(tokens)


# Randomly hide 75% of the patches
visible_tokens, mask, ids_restore = random_masking(
    tokens,
    mask_ratio=0.75,
)

print("Total patches:", mask.shape[1])
print("Visible patches:", int((mask[0] == 0).sum()))
print("Masked patches:", int(mask[0].sum()))


# Apply the SAME mask to the original spectrogram patches
raw_patches = patchify(
    frame,
    patch_size=(4, 64),
)

masked_patches = raw_patches.clone()

masked_patches[
    mask.bool()
] = 0.0


# Turn patches back into an image
masked_frame = unpatchify(
    masked_patches,
    frame_shape=(16, 4096),
    patch_size=(4, 64),
).squeeze().numpy()

original_frame = frame.squeeze().numpy()


# Visualise
fig, axes = plt.subplots(
    2,
    1,
    figsize=(14, 5),
    sharex=True,
)

vmin, vmax = np.percentile(
    original_frame,
    [1, 99],
)

axes[0].imshow(
    original_frame,
    aspect="auto",
    origin="lower",
    vmin=vmin,
    vmax=vmax,
)

axes[0].set_title(
    "Original real radio spectrogram"
)
axes[0].set_ylabel("Time bin")


axes[1].imshow(
    masked_frame,
    aspect="auto",
    origin="lower",
    vmin=vmin,
    vmax=vmax,
)

axes[1].set_title(
    "MAE input — 75% of patches hidden"
)

axes[1].set_ylabel("Time bin")
axes[1].set_xlabel("Frequency channel")

plt.tight_layout()
plt.show()

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# Load separate NORMAL validation observations
raw_val, val_metadata = load_split_frame_pool(
    manifest_path="../configs/data/observations.csv",
    raw_root="../data/raw",
    split="validation",
    f_start=2200,
    f_stop=2300,
)

val_frames = normalize_frames(raw_val)

# Convert NumPy arrays:
# (N, 16, 4096)
#
# into PyTorch image-like tensors:
# (N, 1, 16, 4096)

train_tensor = torch.tensor(
    train_frames,
    dtype=torch.float32,
).unsqueeze(1)

val_tensor = torch.tensor(
    val_frames,
    dtype=torch.float32,
).unsqueeze(1)

train_dataset = TensorDataset(
    train_tensor
)

val_dataset = TensorDataset(
    val_tensor
)

generator = torch.Generator()
generator.manual_seed(42)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    generator=generator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
)

print("Train tensor:", train_tensor.shape)
print("Validation tensor:", val_tensor.shape)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

first_batch = next(iter(train_loader))[0]

print("First batch:", first_batch.shape)

In [ ]:

import importlib
import sys

import deepseti.models.vit_mae as vit_mae

print("Python:", sys.executable)
print("Loaded from:", vit_mae.__file__)
print("Before reload:", hasattr(vit_mae, "ViTMAE"))

importlib.invalidate_caches()
vit_mae = importlib.reload(vit_mae)

print("After reload:", hasattr(vit_mae, "ViTMAE"))
print(
    "Masked loss:",
    hasattr(vit_mae, "masked_reconstruction_loss"),
)

In [ ]:
from deepseti.models.vit_mae import (
    ViTMAE,
    masked_reconstruction_loss,
)

torch.manual_seed(42)

model = ViTMAE(
    frame_shape=(16, 4096),
    patch_size=(4, 64),
    embed_dim=128,
    encoder_depth=4,
    encoder_heads=4,
    encoder_mlp_dim=256,
    decoder_dim=64,
    decoder_depth=2,
    decoder_heads=4,
    decoder_mlp_dim=128,
    mask_ratio=0.75,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.05,
)

num_parameters = sum(
    p.numel()
    for p in model.parameters()
)

print("Device:", device)
print(f"Parameters: {num_parameters:,}")

batch = next(iter(train_loader))[0].to(device)

model.eval()

with torch.no_grad():
    predicted, target, mask = model(batch)

    loss = masked_reconstruction_loss(
        predicted,
        target,
        mask,
    )

print("Input batch:", batch.shape)
print("Predicted patches:", predicted.shape)
print(
    "Masked patches/frame:",
    int(mask[0].sum().item()),
)
print("Initial loss:", loss.item())
print(
    "Finite loss:",
    torch.isfinite(loss).item(),
)

In [ ]:
model.train()

batch = next(iter(train_loader))[0].to(device)

optimizer.zero_grad()

predicted, target, mask = model(batch)

loss_before = masked_reconstruction_loss(
    predicted,
    target,
    mask,
)

loss_before.backward()

gradient_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=1.0,
)

optimizer.step()

print(
    "Training loss:",
    loss_before.detach().item(),
)

print(
    "Gradient norm:",
    gradient_norm.detach().item(),
)

print(
    "Finite loss:",
    torch.isfinite(loss_before).item(),
)

print(
    "Finite gradient norm:",
    torch.isfinite(gradient_norm).item(),
)

In [ ]:
# -------------------------
# Training
# -------------------------

model.train()

train_loss_sum = 0.0

for (batch,) in train_loader:
    batch = batch.to(device)

    optimizer.zero_grad()

    predicted, target, mask = model(batch)

    loss = masked_reconstruction_loss(
        predicted,
        target,
        mask,
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0,
    )

    optimizer.step()

    train_loss_sum += (
        loss.detach().item()
        * batch.size(0)
    )

train_loss = train_loss_sum / len(train_dataset)


# -------------------------
# Validation
# -------------------------

model.eval()

val_loss_sum = 0.0

with torch.no_grad():

    for (batch,) in val_loader:
        batch = batch.to(device)

        predicted, target, mask = model(batch)

        loss = masked_reconstruction_loss(
            predicted,
            target,
            mask,
        )

        val_loss_sum += (
            loss.item()
            * batch.size(0)
        )

val_loss = val_loss_sum / len(val_dataset)


print(f"Epoch 1 train loss: {train_loss:.6f}")
print(f"Epoch 1 val loss:   {val_loss:.6f}")


In [ ]:
from pathlib import Path

checkpoint_dir = Path(
    "../results/checkpoints"
)
checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True,
)

checkpoint_path = (
    checkpoint_dir
    / "vit_mae_best.pt"
)

train_history = [
    train_loss
]

val_history = [
    val_loss
]

best_val_loss = val_loss

torch.save(
    model.state_dict(),
    checkpoint_path,
)

print(
    f"Starting from epoch 1. "
    f"Best val loss: {best_val_loss:.6f}"
)

In [ ]:
for epoch in range(2, 11):

    # -------------------------
    # Training
    # -------------------------

    model.train()

    train_loss_sum = 0.0

    for (batch,) in train_loader:
        batch = batch.to(device)

        optimizer.zero_grad()

        predicted, target, mask = model(
            batch
        )

        loss = masked_reconstruction_loss(
            predicted,
            target,
            mask,
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        optimizer.step()

        train_loss_sum += (
            loss.detach().item()
            * batch.size(0)
        )

    train_loss = (
        train_loss_sum
        / len(train_dataset)
    )


    # -------------------------
    # Validation
    # -------------------------

    model.eval()

    val_loss_sum = 0.0

    with torch.no_grad():

        for (batch,) in val_loader:
            batch = batch.to(device)

            predicted, target, mask = model(
                batch
            )

            loss = masked_reconstruction_loss(
                predicted,
                target,
                mask,
            )

            val_loss_sum += (
                loss.item()
                * batch.size(0)
            )

    val_loss = (
        val_loss_sum
        / len(val_dataset)
    )


    # -------------------------
    # Record history
    # -------------------------

    train_history.append(
        train_loss
    )

    val_history.append(
        val_loss
    )


    # -------------------------
    # Save best model
    # -------------------------

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            checkpoint_path,
        )

        marker = " ← best"

    else:
        marker = ""


    print(
        f"Epoch {epoch:02d} | "
        f"train {train_loss:.6f} | "
        f"val {val_loss:.6f}"
        f"{marker}"
    )

In [ ]:
epochs = range(
    1,
    len(train_history) + 1,
)

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    epochs,
    train_history,
    marker="o",
    label="Train",
)

plt.plot(
    epochs,
    val_history,
    marker="o",
    label="Validation",
)

plt.xlabel("Epoch")
plt.ylabel("Masked reconstruction loss")
plt.title("ViT-MAE training history")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print(
    "Best validation loss:",
    min(val_history),
)

print(
    "Best epoch:",
    np.argmin(val_history) + 1,
)

In [ ]:
for epoch in range(11, 31):

    # -------------------------
    # Training
    # -------------------------
    model.train()

    train_loss_sum = 0.0

    for (batch,) in train_loader:
        batch = batch.to(device)

        optimizer.zero_grad()

        predicted, target, mask = model(batch)

        loss = masked_reconstruction_loss(
            predicted,
            target,
            mask,
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        optimizer.step()

        train_loss_sum += (
            loss.detach().item()
            * batch.size(0)
        )

    train_loss = (
        train_loss_sum
        / len(train_dataset)
    )


    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    val_loss_sum = 0.0

    with torch.no_grad():

        for (batch,) in val_loader:
            batch = batch.to(device)

            predicted, target, mask = model(batch)

            loss = masked_reconstruction_loss(
                predicted,
                target,
                mask,
            )

            val_loss_sum += (
                loss.item()
                * batch.size(0)
            )

    val_loss = (
        val_loss_sum
        / len(val_dataset)
    )


    # -------------------------
    # History
    # -------------------------
    train_history.append(train_loss)
    val_history.append(val_loss)


    # -------------------------
    # Best checkpoint
    # -------------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            checkpoint_path,
        )

        marker = " ← best"

    else:
        marker = ""


    print(
        f"Epoch {epoch:02d} | "
        f"train {train_loss:.6f} | "
        f"val {val_loss:.6f}"
        f"{marker}"
    )

In [ ]:
for epoch in range(31, 51):

    # -------------------------
    # Training
    # -------------------------
    model.train()

    train_loss_sum = 0.0

    for (batch,) in train_loader:
        batch = batch.to(device)

        optimizer.zero_grad()

        predicted, target, mask = model(batch)

        loss = masked_reconstruction_loss(
            predicted,
            target,
            mask,
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        optimizer.step()

        train_loss_sum += (
            loss.detach().item()
            * batch.size(0)
        )

    train_loss = (
        train_loss_sum
        / len(train_dataset)
    )


    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    val_loss_sum = 0.0

    with torch.no_grad():

        for (batch,) in val_loader:
            batch = batch.to(device)

            predicted, target, mask = model(batch)

            loss = masked_reconstruction_loss(
                predicted,
                target,
                mask,
            )

            val_loss_sum += (
                loss.item()
                * batch.size(0)
            )

    val_loss = (
        val_loss_sum
        / len(val_dataset)
    )


    # -------------------------
    # History
    # -------------------------
    train_history.append(train_loss)
    val_history.append(val_loss)


    # -------------------------
    # Best checkpoint
    # -------------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            checkpoint_path,
        )

        marker = " ← best"

    else:
        marker = ""


    print(
        f"Epoch {epoch:02d} | "
        f"train {train_loss:.6f} | "
        f"val {val_loss:.6f}"
        f"{marker}"
    )

In [ ]:
epoch50_path = (
    checkpoint_dir
    / "vit_mae_epoch50_training_state.pt"
)

torch.save(
    {
        "epoch": 50,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_history": train_history,
        "val_history": val_history,
    },
    epoch50_path,
)

print("Saved:", epoch50_path)

In [ ]:
repeated_val_losses = []

model.eval()

with torch.no_grad():

    for repeat in range(5):

        total_loss = 0.0

        for (batch,) in val_loader:

            batch = batch.to(device)

            predicted, target, mask = model(
                batch
            )

            loss = masked_reconstruction_loss(
                predicted,
                target,
                mask,
            )

            total_loss += (
                loss.item()
                * batch.size(0)
            )

        repeat_loss = (
            total_loss
            / len(val_dataset)
        )

        repeated_val_losses.append(
            repeat_loss
        )

        print(
            f"Mask run {repeat + 1}: "
            f"{repeat_loss:.6f}"
        )


print()
print(
    "Mean validation loss:",
    np.mean(repeated_val_losses),
)

print(
    "Std validation loss:",
    np.std(repeated_val_losses),
)

In [ ]:
for epoch in range(51, 76):

    # -------------------------
    # Training
    # -------------------------
    model.train()

    train_loss_sum = 0.0

    for (batch,) in train_loader:
        batch = batch.to(device)

        optimizer.zero_grad()

        predicted, target, mask = model(batch)

        loss = masked_reconstruction_loss(
            predicted,
            target,
            mask,
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        optimizer.step()

        train_loss_sum += (
            loss.detach().item()
            * batch.size(0)
        )

    train_loss = (
        train_loss_sum
        / len(train_dataset)
    )


    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    val_loss_sum = 0.0

    with torch.no_grad():

        for (batch,) in val_loader:
            batch = batch.to(device)

            predicted, target, mask = model(batch)

            loss = masked_reconstruction_loss(
                predicted,
                target,
                mask,
            )

            val_loss_sum += (
                loss.item()
                * batch.size(0)
            )

    val_loss = (
        val_loss_sum
        / len(val_dataset)
    )


    train_history.append(train_loss)
    val_history.append(val_loss)


    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            checkpoint_path,
        )

        marker = " ← best"

    else:
        marker = ""


    print(
        f"Epoch {epoch:02d} | "
        f"train {train_loss:.6f} | "
        f"val {val_loss:.6f}"
        f"{marker}"
    )

In [ ]:
for epoch in range(76, 101):

    model.train()
    train_loss_sum = 0.0

    for (batch,) in train_loader:
        batch = batch.to(device)

        optimizer.zero_grad()

        predicted, target, mask = model(batch)

        loss = masked_reconstruction_loss(
            predicted,
            target,
            mask,
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        optimizer.step()

        train_loss_sum += (
            loss.detach().item()
            * batch.size(0)
        )

    train_loss = (
        train_loss_sum
        / len(train_dataset)
    )


    model.eval()
    val_loss_sum = 0.0

    with torch.no_grad():

        for (batch,) in val_loader:
            batch = batch.to(device)

            predicted, target, mask = model(batch)

            loss = masked_reconstruction_loss(
                predicted,
                target,
                mask,
            )

            val_loss_sum += (
                loss.item()
                * batch.size(0)
            )

    val_loss = (
        val_loss_sum
        / len(val_dataset)
    )


    train_history.append(train_loss)
    val_history.append(val_loss)


    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            checkpoint_path,
        )

        marker = " ← best"

    else:
        marker = ""


    print(
        f"Epoch {epoch:03d} | "
        f"train {train_loss:.6f} | "
        f"val {val_loss:.6f}"
        f"{marker}"
    )

In [ ]:
epoch100_path = (
    checkpoint_dir
    / "vit_mae_epoch100_training_state.pt"
)

torch.save(
    {
        "epoch": 100,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_history": train_history,
        "val_history": val_history,
        "best_val_loss": best_val_loss,
    },
    epoch100_path,
)

print("Saved:", epoch100_path)
print("Best validation loss:", best_val_loss)

In [ ]:
repeated_val_losses_100 = []

model.eval()

with torch.no_grad():

    for repeat in range(5):

        total_loss = 0.0

        for (batch,) in val_loader:
            batch = batch.to(device)

            predicted, target, mask = model(batch)

            loss = masked_reconstruction_loss(
                predicted,
                target,
                mask,
            )

            total_loss += (
                loss.item()
                * batch.size(0)
            )

        repeat_loss = (
            total_loss
            / len(val_dataset)
        )

        repeated_val_losses_100.append(
            repeat_loss
        )

        print(
            f"Mask run {repeat + 1}: "
            f"{repeat_loss:.6f}"
        )

print()
print(
    "Mean:",
    np.mean(repeated_val_losses_100),
)

print(
    "Std:",
    np.std(repeated_val_losses_100),
)

In [ ]:
for epoch in range(101, 126):

    model.train()
    train_loss_sum = 0.0

    for (batch,) in train_loader:
        batch = batch.to(device)

        optimizer.zero_grad()

        predicted, target, mask = model(batch)

        loss = masked_reconstruction_loss(
            predicted,
            target,
            mask,
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        optimizer.step()

        train_loss_sum += (
            loss.detach().item()
            * batch.size(0)
        )

    train_loss = (
        train_loss_sum
        / len(train_dataset)
    )


    model.eval()
    val_loss_sum = 0.0

    with torch.no_grad():

        for (batch,) in val_loader:
            batch = batch.to(device)

            predicted, target, mask = model(batch)

            loss = masked_reconstruction_loss(
                predicted,
                target,
                mask,
            )

            val_loss_sum += (
                loss.item()
                * batch.size(0)
            )

    val_loss = (
        val_loss_sum
        / len(val_dataset)
    )

    train_history.append(train_loss)
    val_history.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            checkpoint_path,
        )

        marker = " ← best"
    else:
        marker = ""

    print(
        f"Epoch {epoch:03d} | "
        f"train {train_loss:.6f} | "
        f"val {val_loss:.6f}"
        f"{marker}"
    )

In [ ]:
epoch125_path = (
    checkpoint_dir
    / "vit_mae_epoch125_training_state.pt"
)

torch.save(
    {
        "epoch": 125,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_history": train_history,
        "val_history": val_history,
        "best_val_loss": best_val_loss,
    },
    epoch125_path,
)

print("Saved:", epoch125_path)
print("Best validation loss:", best_val_loss)

In [ ]:
epoch100_path = (
    checkpoint_dir
    / "vit_mae_epoch100_training_state.pt"
)

print("Epoch 100 exists:", epoch100_path.exists())
print("Epoch 125 exists:", epoch125_path.exists())